In [56]:
from ucimlrepo import fetch_ucirepo 
  
student_performance = fetch_ucirepo(id=320) 
  
X = student_performance.data.features
y = student_performance.data.targets

X["G1"], X["G2"] = y["G1"], y["G2"]
y = y.drop(columns = ["G1", "G2"])

In [57]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.3, random_state = 1
)

In [58]:
nominalColumns = ["Mjob", "Fjob", "reason", "guardian"]
ordinalColumns = ["Medu", "Fedu", "traveltime", "studytime", "famrel", "freetime",
                  "goout", "Dalc", "Walc", "health"]
binaryColumns = ["school", "sex", "address", "famsize", "Pstatus", "schoolsup", "famsup",
                 "paid", "activities", "nursery", "higher", "internet", "romantic",]
numericColumns = ["age", "absences", "failures", "G1", "G2"]

In [59]:
import sklearn.preprocessing as preprocessing
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

nominalPipe = Pipeline([
    ("imputer", SimpleImputer(strategy = "most_frequent")),
    ("onehot", preprocessing.OneHotEncoder(handle_unknown = "ignore", sparse_output = False))
])

ordinalPipe = Pipeline([
    ("imputer", SimpleImputer(strategy = "most_frequent")),
    ("scaler", preprocessing.StandardScaler()),
])

binaryPipe = Pipeline([
    ("imputer", SimpleImputer(strategy = "most_frequent")),
    ("onehot", preprocessing.OneHotEncoder(drop = "if_binary", handle_unknown = "ignore",
                                           sparse_output = False))
])

numericPipe = Pipeline([
    ("imputer", SimpleImputer(strategy = "median")),
    ("scaler", preprocessing.StandardScaler())
])

In [60]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers = [
        ("nominal", nominalPipe, nominalColumns),
        ("ordinal", ordinalPipe, ordinalColumns),
        ("binary", binaryPipe, binaryColumns),
        ("numeric", numericPipe, numericColumns)
    ],
    remainder = "drop",
    verbose_feature_names_out = False
).set_output(transform = "pandas")

In [61]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(model.score(X_test, y_test))
print(mean_absolute_error(y_test, y_pred))

0.8480671338660938
0.8340183874257263
